# Suite 2: FullStory-Bench — ConStory-Bench + EQ-Bench Longform + DOC-NLI

Đánh giá **truyện dài đầy đủ (8.000–10.000 từ)** trực tiếp từ file JSONL đã lưu trên Google Drive (mặc định cắm sẵn vào kết quả của `ablation_context_gate_validator` hoặc `Base 7B vs GRPO 7B`):
- **Phase 1 (Deterministic + DeBERTa-v3 NLI + MiniLM)**:
  - Length Compliance (`Sl %`), Early Termination / Overshoot, `<think>` word budget
  - DOC-NLI Plan Adherence (`S_adh`)
  - Inter-Chapter Continuity (`transition_cosine`), Anti-Context Novelty & Fallback Rate
  - Lexical Richness (`Distinct-3/4`, Entropy, Burstiness), **Slop Score / 1k words**, **Intra-Chapter Redundancy**
- **Phase 2 (Judge LLM — `local_vllm` hoặc `api`)**:
  - **ConStory-Checker (ACL Findings 2026)**: 5 nhóm / 19 loại lỗi mâu thuẫn kèm trích dẫn nguyên văn (`errors_per_1k`, `C_weighted`)
  - **EQ-Bench Longform**: 6 tiêu chí (0–10) gồm `meaningful_elaboration` + Early-vs-Late `degradation_penalty` → `eqbench_score` (0–100)
  - **Blind Pairwise Judge**: So sánh mù A/B đảo chiều loại bỏ position bias.

In [ ]:
# 1. Mount Google Drive & Sync Repository
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = Path('/content/MA-GRPO')
REPO_URL = 'github.com/AIVIETNAM-AIO-TruongDogki/MA_GRPO.git'
try:
    from google.colab import userdata
    GH_TOKEN = userdata.get('GH_TOKEN') or os.environ.get('GH_TOKEN', '')
except Exception:
    GH_TOKEN = os.environ.get('GH_TOKEN', '')

clone_url = f'https://{GH_TOKEN}@{REPO_URL}' if GH_TOKEN else f'https://{REPO_URL}'
if not REPO_DIR.exists():
    !git clone {clone_url} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull --ff-only
%cd {REPO_DIR}

In [ ]:
# 2. Install Lightweight Dependencies
!pip install -q transformers sentence-transformers accelerate openai matplotlib pandas

In [ ]:
# 3. Configure Story Inputs (Pre-wired to ContextGate Ablation Output on Drive)
ABLATION_DIR = '/content/drive/MyDrive/MA-GRPO-ContextGate-Ablation'
STORIES_FILES = [
    f'{ABLATION_DIR}/stories_7b_outputs.jsonl',
]
TRANSITIONS_FILE = f'{ABLATION_DIR}/coordinator_transitions.jsonl'
OUTPUT_DIR = '/content/drive/MyDrive/MA-GRPO-Benchmarks/suite2_fullstory_bench'

# Judge backend: 'none' | 'local_vllm' (chấm ConStory + EQ-Bench bằng 32B-AWQ trên A100) | 'api'
JUDGE_BACKEND = 'local_vllm'
JUDGE_MODEL = 'Qwen/Qwen2.5-32B-Instruct-AWQ'
JUDGE_MAX_LEN = 24576

# So sánh mù hai nhánh của ablation (hoặc đổi thành ('base_7b', 'grpo_7b') sau khi train GRPO)
PAIRWISE_ARMS = ('validator_0.80', 'no_validator')
AUTO_DISCONNECT = False

if JUDGE_BACKEND == 'local_vllm':
    import importlib.util
    if importlib.util.find_spec('vllm') is None:
        !pip install -q vllm

In [ ]:
# 4. Execute Suite 2 (FullStory-Bench) via Clean Subprocess
import subprocess
import sys

cmd = [
    sys.executable,
    'scripts/benchmark_fullstory.py',
    '--stories', *STORIES_FILES,
    '--output-dir', OUTPUT_DIR,
    '--judge-backend', JUDGE_BACKEND,
    '--judge-model', JUDGE_MODEL,
    '--judge-max-len', str(JUDGE_MAX_LEN),
]
if TRANSITIONS_FILE and Path(TRANSITIONS_FILE).exists():
    cmd.extend(['--transitions', TRANSITIONS_FILE])
if PAIRWISE_ARMS:
    cmd.extend(['--pairwise-arms', PAIRWISE_ARMS[0], PAIRWISE_ARMS[1]])

env = os.environ.copy()
env['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
env['VLLM_LOGGING_LEVEL'] = 'WARNING'
env['PYTHONUNBUFFERED'] = '1'

print('Running:', ' '.join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
for line in proc.stdout:
    print(line, end='', flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'benchmark_fullstory.py exited with code {ret}')

In [ ]:
# 5. Display Report, ConStory Error Breakdown, Pairwise Verdicts & 4-Panel Chart
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

out_path = Path(OUTPUT_DIR)
df = pd.read_csv(out_path / 'fullstory_summary.csv')
display(df)
display(Markdown((out_path / 'fullstory_benchmark_report.md').read_text(encoding='utf-8')))

# Preview top ConStory-Checker errors with quotes
err_file = out_path / 'constory_errors.jsonl'
if err_file.exists():
    errs = [json.loads(l) for l in err_file.read_text(encoding='utf-8').splitlines() if l.strip()]
    print(f'Total ConStory-Checker evidence errors logged: {len(errs)}')
    if errs:
        display(pd.DataFrame(errs)[['arm', 'premise_id', 'category', 'subtype', 'severity', 'chapter_a', 'chapter_b', 'explanation']].head(15))

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
panels = [
    ('S_adh', 'DOC-NLI Adherence (S_adh)', (0, 1.05)),
    ('transition_cosine', 'Inter-Chapter Continuity', (0, 1.05)),
    ('C_weighted', 'ConStory C_weighted (Lower=Better)', None),
    ('eqbench_score', 'EQ-Bench Longform (0-100)', (0, 105)),
]
for ax, (col, title, ylim) in zip(axes, panels):
    vals = df[col].fillna(0)
    ax.bar(df['arm'], vals, color='#2f855a')
    if ylim:
        ax.set_ylim(*ylim)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(out_path / 'fullstory_benchmark_chart.png', dpi=150)
plt.show()

if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()